# 3.3 หา Insight (ฉบับแก้): เปรียบเทียบรีวิวคะแนนต่ำ vs คะแนนสูง
ใช้ต่อจากตาราง `corp` ใน notebook ของคุณ (หรืออ่านจาก `tmdb_corpus_clean.csv`) ทดสอบกับข้อมูลจริงที่คุณแนบมาแล้ว

**คำถามที่ใช้ (ใส่ในช่อง Question ของ 3.1 ได้เลย):** ผู้ชมที่ไม่พอใจซีรีส์ใช้คำ/พูดถึงเรื่องอะไรต่างจากผู้ชมที่พอใจ เพื่อให้ทีมจัดหาคอนเทนต์ใช้เป็นสัญญาณเตือนตอนคัดซีรีส์

**เทคนิค:** comparison ระหว่างกลุ่ม (log-odds ของคำ) ร่วมกับ frequency โดยเก็บคำปฏิเสธ และนับจำนวนซีรีส์ที่ต่างกันที่คำนั้นปรากฏ

**ที่เปลี่ยนจากเวอร์ชันเดิม**
| เดิม | ใหม่ | เหตุผล |
|---|---|---|
| วิเคราะห์ 30 รีวิว (ซีรีส์ดัง + คะแนน ≤5) | ใช้รีวิวที่มีคะแนนทั้งหมด แบ่งต่ำ (≤4) / สูง (≥9) | n มากพอ และมีกลุ่มเปรียบเทียบ |
| `not/no/never` ถูกตัดเป็น stopword | แปลง "not funny" เป็น `neg_funny` | ไม่ให้ความหมายกลับด้าน |
| ลบชื่อซีรีส์ทุกเรื่องออกจากทุกรีวิว | ลบเฉพาะชื่อของซีรีส์ที่รีวิวนั้นพูดถึง | กันลบคำสามัญ เช่น house, lost, you |
| NMF 4 หัวข้อบน 30 เอกสาร | log-odds + จำนวนซีรีส์ | โมเดลหัวข้อไม่นิ่งเมื่อเอกสารน้อย |

## 1) Setup และเตรียมกลุ่ม
**ทำอะไร:** อ่าน corpus, ตัดรีวิวที่ไม่มีคะแนน (พร้อมรายงานจำนวน), แบ่งกลุ่มตามคะแนน \
**ทำไมไม่เอากลุ่มกลาง (5–8):** รีวิวคะแนนกลางมักปนทั้งชมและติ การเทียบปลายสองด้านทำให้ความต่างชัดและตีความง่าย

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

CORPUS_CSV = "tmdb_corpus_clean.csv"      # ถ้ามี DATA_DIR ให้ใช้ CORPUS_CSV ที่ประกาศใน Setup แทน
try:
    df_all = corp.copy()                  # ใช้ตาราง corp จากขั้น Clean ถ้ายังอยู่ในหน่วยความจำ
except NameError:
    df_all = pd.read_csv(CORPUS_CSV)

LOW_MAX, HIGH_MIN = 4, 9                  # ต่ำ = คะแนน <= 4, สูง = คะแนน >= 9 (เต็ม 10)
MIN_DOCS = 5                              # คำต้องอยู่ในอย่างน้อย 5 รีวิว
MIN_SERIES = 4                            # และมาจากอย่างน้อย 4 ซีรีส์ที่ต่างกัน

df_all["user_rating"] = pd.to_numeric(df_all["user_rating"], errors="coerce")
n_norating = int(df_all["user_rating"].isna().sum())
df = df_all.dropna(subset=["user_rating"]).copy()
df["group"] = np.select([df["user_rating"] <= LOW_MAX, df["user_rating"] >= HIGH_MIN],
                        ["low", "high"], default="mid")

print(f"Corpus {len(df_all)} รีวิว | ไม่มี user_rating {n_norating} ({n_norating / len(df_all) * 100:.1f}%) -> ใช้ได้ {len(df)} รีวิว")
summary = (df.groupby("group").agg(reviews=("review_id", "size"), series=("doc_id", "nunique"))
             .reindex(["low", "mid", "high"]))
display(summary)

## 2) ล้างข้อความโดย **เก็บคำปฏิเสธ**
**ทำอะไร (เรียงตามลำดับในฟังก์ชัน)**
1. ตัวพิมพ์เล็ก, แปลง ’ เป็น ', ตัด URL และเครื่องหมาย markdown (`**`, `##`) ที่พบในรีวิวจริง
2. ลบชื่อซีรีส์ของรีวิวนั้น (ใช้ขอบเขตคำ ไม่ลบคำที่บังเอิญเหมือนชื่อ)
3. ขยาย "don't" → "do not", "can't" → "can not"
4. **แปลง "not/no/never + คำถัดไป" เป็นโทเค็นเดียว** เช่น "is not funny" → `neg_funny` โดยข้ามคำเชื่อมอย่าง a, the, very และไม่ข้ามเครื่องหมายวรรคตอน (กันการเชื่อมข้ามประโยค)
5. ตัดอักขระที่ไม่ใช่ตัวอักษร, ตัด stopword, ตัดคำที่สั้นกว่า 3 ตัว

**ทำไมต้องทำขั้น 4 ก่อนตัด stopword:** รายการ stopword ของ scikit-learn มี `not`, `no`, `never` อยู่ด้วย ถ้าตัดก่อน "not funny" จะเหลือ "funny"

In [ ]:
CUSTOM_STOP = {"show", "shows", "series", "season", "seasons", "episode", "episodes", "watch", "watched",
               "watching", "tv", "really", "just", "like", "would", "could", "also", "get", "got", "make",
               "made", "thing", "things", "even", "still", "much", "many", "lot", "one", "bit"}
STOP = set(ENGLISH_STOP_WORDS) | CUSTOM_STOP

SKIP = r"(?:a|an|the|very|so|too|really|that|even|as|be|been|being|is|are|was|were|do|does|did|have|has|had|to|just|much|all|quite)"
NEG_RE = re.compile(rf"\b(?:not|no|never|hardly|barely)\s+(?:{SKIP}\s+)*([a-z]+)")

def clean_review(text, title=""):
    t = str(text).lower().replace("’", "'").replace("‘", "'")
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)                  # URL
    t = re.sub(r"[*#_>`]+", " ", t)                               # markdown
    if title:                                                     # ลบชื่อซีรีส์ของรีวิวนี้เท่านั้น
        t = re.sub(r"(?<!\w)" + re.escape(str(title).lower()) + r"(?!\w)", " ", t)
    t = (t.replace("can't", "can not").replace("cannot", "can not")
           .replace("won't", "will not").replace("n't", " not"))   # ขยายคำย่อปฏิเสธ
    t = re.sub(r"'(?:s|re|ve|ll|d|m)\b", " ", t)                  # 's 're 've 'll 'd 'm
    t = NEG_RE.sub(lambda m: " " if m.group(1) in STOP else f" neg_{m.group(1)} ", t)
    t = re.sub(r"[^a-z_\s]", " ", t)                              # เหลือตัวอักษรกับ _
    return " ".join(w for w in t.split() if len(w) >= 3 and w not in STOP)

df["clean"] = [clean_review(t, n) for t, n in zip(df["text"], df["name"])]
n_before = len(df)
df = df[df["clean"].str.strip() != ""].copy()
print(f"ตัดรีวิวที่ว่างหลังล้างข้อความ {n_before - len(df)} -> เหลือ {len(df)}")

# ตรวจด้วยตา: ตัวอย่างที่มีคำปฏิเสธ
ex = df[df["clean"].str.contains("neg_")].sample(3, random_state=RANDOM_SEED)
for _, r in ex.iterrows():
    print(f"\n[{r['name']}] คะแนน {r['user_rating']:.0f}")
    print("  ต้นฉบับ:", re.sub(r"\s+", " ", r["text"])[:160])
    print("  หลังล้าง:", r["clean"][:160])

## 3) เปรียบเทียบคำระหว่างกลุ่มต่ำ vs สูง
**ทำอะไร:** นับว่าแต่ละคำปรากฏใน "กี่รีวิว" ของแต่ละกลุ่ม (นับครั้งเดียวต่อรีวิว) แล้วคำนวณ log-odds ratio พร้อมค่า z \
**ตีความ:** `z > 0` คำนี้พบในรีวิวคะแนนต่ำมากกว่า, `z < 0` พบในรีวิวคะแนนสูงมากกว่า \
**ทำไมนับ "จำนวนซีรีส์":** คำที่มาจากซีรีส์เดียว (เช่น ชื่อตัวละคร) ไม่ใช่ข้อบ่นทั่วไป จึงกรองให้เหลือคำที่มาจาก ≥ `MIN_SERIES` ซีรีส์ \
**ค่า z ใช้จัดอันดับเท่านั้น:** ทดสอบหลายร้อยคำพร้อมกัน จึงไม่ควรอ้างเป็นนัยสำคัญทางสถิติ ต้องอ่านรีวิวจริงยืนยันทุกครั้ง (ดูขั้น 6)

In [ ]:
def contrast(d, low_max=LOW_MAX, high_min=HIGH_MIN, min_docs=MIN_DOCS):
    d = d[(d["user_rating"] <= low_max) | (d["user_rating"] >= high_min)].reset_index(drop=True)
    y = (d["user_rating"] <= low_max).values                       # True = กลุ่มต่ำ

    cv = CountVectorizer(binary=True, lowercase=False, min_df=min_docs, token_pattern=r"[a-z_]{3,}")
    X = cv.fit_transform(d["clean"]).tocsr()
    terms = cv.get_feature_names_out()

    codes = pd.factorize(d["doc_id"])[0]                           # รหัสซีรีส์ของแต่ละรีวิว
    S = csr_matrix((np.ones(len(d)), (codes, np.arange(len(d)))), shape=(codes.max() + 1, len(d)))

    def group_stats(mask):
        idx = np.where(mask)[0]
        Xg, Sg = X[idx], S[:, idx]
        docs = np.asarray(Xg.sum(axis=0)).ravel()                  # จำนวนรีวิวที่มีคำนี้
        per_series = (Sg @ Xg).toarray()                           # ซีรีส์ x คำ
        n_series = (per_series > 0).sum(axis=0)                    # จำนวนซีรีส์ที่ต่างกัน
        top_share = per_series.max(axis=0) / np.maximum(docs, 1)   # สัดส่วนที่มาจากซีรีส์เดียว
        return docs, n_series, top_share

    a, sa, ta = group_stats(y)
    b, sb, tb = group_stats(~y)
    nl, nh = int(y.sum()), int((~y).sum())
    lor = np.log((a + .5) / (nl - a + .5)) - np.log((b + .5) / (nh - b + .5))
    se = np.sqrt(1 / (a + .5) + 1 / (nl - a + .5) + 1 / (b + .5) + 1 / (nh - b + .5))

    out = pd.DataFrame({
        "term": [("not " + t[4:]) if t.startswith("neg_") else t for t in terms],
        "low_reviews": a, "high_reviews": b,
        "low_pct": a / nl * 100, "high_pct": b / nh * 100,
        "low_series": sa, "high_series": sb,
        "low_top_series_%": ta * 100, "high_top_series_%": tb * 100,
        "z": lor / se,
    })
    return out, nl, nh

def pick_terms(res, side, k=15, min_docs=MIN_DOCS, min_series=MIN_SERIES):
    if side == "low":
        r = res[(res["low_reviews"] >= min_docs) & (res["low_series"] >= min_series) & (res["z"] > 0)]
        return r.sort_values("z", ascending=False).head(k)
    r = res[(res["high_reviews"] >= min_docs) & (res["high_series"] >= min_series) & (res["z"] < 0)]
    return r.sort_values("z").head(k)

res, n_low, n_high = contrast(df)
low_terms, high_terms = pick_terms(res, "low"), pick_terms(res, "high")

print(f"กลุ่มต่ำ n = {n_low} รีวิว | กลุ่มสูง n = {n_high} รีวิว | คำที่นำมาเทียบ {len(res)} คำ")
cols = ["term", "low_reviews", "high_reviews", "low_pct", "high_pct", "low_series", "z"]
print("\nคำที่พบมากกว่าในรีวิวคะแนนต่ำ:");  display(low_terms[cols].round(2).reset_index(drop=True))
cols_h = ["term", "high_reviews", "low_reviews", "high_pct", "low_pct", "high_series", "z"]
print("คำที่พบมากกว่าในรีวิวคะแนนสูง:");   display(high_terms[cols_h].round(2).reset_index(drop=True))

## 4) กราฟเปรียบเทียบ (2 รูป)
**รูปที่ 1 — ขนาดกลุ่ม:** บอกผู้ชมก่อนว่าแต่ละกลุ่มมีกี่รีวิว/กี่ซีรีส์ (กราฟที่ไม่มี n น่าเชื่อถือน้อย) \
**รูปที่ 2 — เปรียบเทียบคำ:** สองช่อง ซ้าย = คำที่เด่นในรีวิวคะแนนต่ำ ขวา = คำที่เด่นในรีวิวคะแนนสูง ในแต่ละคำมี 2 แท่ง คือ % ของรีวิวในกลุ่มต่ำ และ % ของรีวิวในกลุ่มสูงที่มีคำนั้น ช่องว่างระหว่างสองแท่งคือความต่างที่เรากำลังพูดถึง \
**ฟอนต์ไทย:** ถ้าไม่พบฟอนต์ไทย โค้ดจะสลับเป็นป้ายภาษาอังกฤษให้เอง บน Colab ติดตั้งด้วย `!apt-get -qq install -y fonts-thai-tlwg` แล้ว Restart runtime

In [ ]:
from matplotlib import font_manager as fm
THAI_FONTS = ["Tahoma", "Leelawadee UI", "Noto Sans Thai", "Sarabun", "TH Sarabun New", "Garuda", "Loma", "Tlwg Typo"]
_avail = {f.name for f in fm.fontManager.ttflist}
_thai = next((f for f in THAI_FONTS if f in _avail), None)
if _thai:
    plt.rcParams["font.family"] = _thai
plt.rcParams.update({"axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False})
T = (lambda th, en: th) if _thai else (lambda th, en: en)      # เลือกป้ายตามฟอนต์ที่มี

C_LOW, C_HIGH, C_MID = "#C0504D", "#2A9D8F", "#B0B0B0"

# ---------- รูปที่ 1: ขนาดกลุ่ม ----------
g = summary.reset_index()
labels = {"low": T(f"ต่ำ (≤{LOW_MAX})", f"Low (≤{LOW_MAX})"),
          "mid": T("กลาง (ไม่ใช้เทียบ)", "Middle (not compared)"),
          "high": T(f"สูง (≥{HIGH_MIN})", f"High (≥{HIGH_MIN})")}
fig, ax = plt.subplots(figsize=(7.5, 3.2))
bars = ax.barh([labels[x] for x in g["group"]], g["reviews"], color=[C_LOW, C_MID, C_HIGH], height=0.55)
for b_, n_s in zip(bars, g["series"]):
    ax.text(b_.get_width() + 3, b_.get_y() + b_.get_height() / 2,
            T(f"{int(b_.get_width())} รีวิว จาก {n_s} ซีรีส์", f"{int(b_.get_width())} reviews from {n_s} series"),
            va="center", fontsize=10)
ax.set_xlim(0, g["reviews"].max() * 1.45)
ax.invert_yaxis()
ax.set_xlabel(T("จำนวนรีวิว", "Number of reviews"))
ax.set_title(T(f"ขนาดแต่ละกลุ่มคะแนน (รวม {len(df)} รีวิวที่มี user_rating)",
               f"Group sizes (total {len(df)} rated reviews)"))
plt.tight_layout(); plt.show()

# ---------- รูปที่ 2: เปรียบเทียบคำ ----------
def draw_panel(ax, table, title, order_col):
    t = table.head(10).iloc[::-1]                       # คำที่เด่นสุดอยู่บน
    y = np.arange(len(t)); h = 0.38
    ax.barh(y + h / 2, t["low_pct"], height=h, color=C_LOW, label=T(f"กลุ่มต่ำ (n={n_low})", f"Low (n={n_low})"))
    ax.barh(y - h / 2, t["high_pct"], height=h, color=C_HIGH, label=T(f"กลุ่มสูง (n={n_high})", f"High (n={n_high})"))
    ax.set_yticks(y)
    ax.set_yticklabels([f"{w}  ({int(s)} {T('ซีรีส์', 'series')})" for w, s in zip(t["term"], t[order_col])], fontsize=10)
    ax.set_title(title, fontsize=12, loc="left")
    ax.set_xlabel(T("% ของรีวิวในกลุ่มที่มีคำนี้", "% of reviews in group containing the term"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
draw_panel(axes[0], low_terms, T("คำที่พบมากกว่าในรีวิวคะแนนต่ำ", "More common in LOW-rated reviews"), "low_series")
draw_panel(axes[1], high_terms, T("คำที่พบมากกว่าในรีวิวคะแนนสูง", "More common in HIGH-rated reviews"), "high_series")
axes[0].legend(loc="lower right", fontsize=9)
fig.suptitle(T("ผู้ชมที่ไม่พอใจกับผู้ชมที่พอใจใช้คำต่างกันอย่างไร (TMDB, ตัวเลขในวงเล็บ = จำนวนซีรีส์ที่พบคำนั้น)",
               "How dissatisfied vs satisfied viewers differ in wording (TMDB; brackets = number of series)"), fontsize=12, x=0.01, ha="left")
plt.tight_layout(); plt.show()

## 5) Reality Check: อ่านผลนี้อย่างไรให้ไม่ overclaim
ตัวเลขข้างบนยังไม่ใช่ข้อสรุป ต้องลองหักล้างด้วย 3 ข้อ

| ทดสอบ | คำถาม | ผ่านถ้า |
|---|---|---|
| ความกระจุกตัว | คำนี้มาจากซีรีส์เดียวเป็นส่วนใหญ่ไหม | `low_top_series_%` ไม่สูงเกินไป (เช่น < 50%) |
| อ่านจริง | คำนี้ถูกใช้ในบริบทที่เราตีความไหม | ตัวอย่างข้อความตรงกับความหมายที่เราเขียน |
| ความทนทาน | เปลี่ยนเกณฑ์คะแนน/กลุ่มซีรีส์แล้วผลยังคล้ายเดิมไหม | คำเด่นซ้อนทับกับผลหลักเป็นส่วนมาก |

In [ ]:
# ---------- ทดสอบ 1: ความกระจุกตัว ----------
chk = low_terms[["term", "low_reviews", "low_series", "low_top_series_%"]].copy()
chk["กระจุกตัว"] = np.where(chk["low_top_series_%"] >= 50, "ระวัง (>=50% จากซีรีส์เดียว)", "ok")
print("ความกระจุกตัวของคำเด่นในรีวิวคะแนนต่ำ:"); display(chk.round(1).reset_index(drop=True))

# ---------- ทดสอบ 2: อ่านบริบทจริง (KWIC) ----------
def show_examples(term, group="low", k=3, width=70):
    base = term[4:] if term.startswith("not ") else term
    sel = df[(df["group"] == group) & df["clean"].str.contains(rf"\b(?:neg_)?{re.escape(base)}\b")]
    print(f"\n=== '{term}' ({group}) พบ {len(sel)} รีวิว ===")
    for _, r in sel.sample(min(k, len(sel)), random_state=RANDOM_SEED).iterrows():
        raw = re.sub(r"\s+", " ", r["text"].replace("’", "'"))
        m = re.search(re.escape(base), raw, flags=re.I)
        s = max(m.start() - width, 0) if m else 0
        print(f"  [{r['name'][:24]} | {r['user_rating']:.0f}/10] ...{raw[s:s + 2 * width]}...")

for w in low_terms["term"].head(3):          # ดู 3 คำแรกก่อน เปลี่ยนเป็นคำอื่นได้
    show_examples(w, "low")

# ---------- ทดสอบ 3: ความทนทานต่อการเปลี่ยนเกณฑ์ ----------
main_top = set(low_terms["term"].head(15))
pop_cut = df.drop_duplicates("doc_id")["popularity"].median()
variants = {
    "ต่ำ<=3 / สูง>=9": (df, 3, 9),
    "ต่ำ<=4 / สูง>=8": (df, 4, 8),
    "ต่ำ<=5 / สูง>=9": (df, 5, 9),
    "เฉพาะซีรีส์ popularity สูงกว่ามัธยฐาน": (df[df["popularity"] >= pop_cut], 4, 9),
}
rows = []
for name, (d_, lo_, hi_) in variants.items():
    r_, nl_, nh_ = contrast(d_, lo_, hi_)
    top_ = set(pick_terms(r_, "low", 15)["term"])
    rows.append({"ตัวแปร": name, "n ต่ำ": nl_, "n สูง": nh_, "คำเด่น (ซ้อนกับผลหลัก จาก 15)": len(main_top & top_)})
print("\nความทนทาน:"); display(pd.DataFrame(rows))

## 6) ตัวเลขสำหรับกรอก Insight Card (3.4)
รันเซลล์นี้แล้วนำตัวเลขไปเขียน **อย่าใช้ตัวเลขจากคำอธิบายในไฟล์นี้** ต้องเป็นผลที่ได้จากข้อมูลและการอ่านรีวิวจริงของคุณ

โครงการ์ด:
> **Insight:** รีวิวคะแนนต่ำ (≤4, n = [..]) พูดถึง/ใช้คำ *[คำ]* ใน [..]% ของรีวิว เทียบ [..]% ในรีวิวคะแนนสูง (≥9, n = [..]) โดยพบใน [..] ซีรีส์ที่ต่างกัน \
> **หลักฐาน:** กราฟเปรียบเทียบคำ (รูปที่ 2) และตารางความกระจุกตัว; ผ่าน/ไม่ผ่านการทดสอบทนทาน [..] \
> **So what:** ถ้าซีรีส์ที่กำลังพิจารณามีสัญญาณ [..] ในรีวิวผู้ชม ควรนับเป็นความเสี่ยงด้านความพึงพอใจ \
> **Action:** ทีมจัดหาคอนเทนต์เพิ่มเกณฑ์ตรวจ [..] ใน shortlist ก่อนเจรจาลิขสิทธิ์ และติดตามสัดส่วนรีวิวที่มีคำนี้ต่อไตรมาส \
> **ความมั่นใจ:** ต่ำ/กลาง เพราะ [n ของกลุ่มต่ำ, ซีรีส์ยอดนิยมภาษาอังกฤษเป็นหลัก, ไม่มี user_rating [..]%, เป็นเสียงของคนที่เขียนรีวิว ไม่ใช่ยอดชม]

**ข้อมูลชุดนี้ยังตอบอะไรไม่ได้:** ขนาดของปัญหาต่อฐานผู้ชมทั้งหมด (รีวิวเป็นกลุ่มที่สมัครใจเขียน), ผลต่อยอดรับชมหรือการยกเลิกสมาชิก, และผู้ชมที่ใช้ภาษาอื่น

In [ ]:
top_l = low_terms.iloc[0]
top_h = high_terms.iloc[0]
print(f"กลุ่มต่ำ n = {n_low} รีวิว ({df[df['group']=='low']['doc_id'].nunique()} ซีรีส์) | กลุ่มสูง n = {n_high} รีวิว ({df[df['group']=='high']['doc_id'].nunique()} ซีรีส์)")
print(f"คำเด่นฝั่งต่ำอันดับ 1: '{top_l['term']}' = {top_l['low_pct']:.1f}% ของรีวิวต่ำ vs {top_l['high_pct']:.1f}% ของรีวิวสูง | {int(top_l['low_series'])} ซีรีส์ | จากซีรีส์เดียวมากสุด {top_l['low_top_series_%']:.0f}%")
print(f"คำเด่นฝั่งสูงอันดับ 1: '{top_h['term']}' = {top_h['high_pct']:.1f}% ของรีวิวสูง vs {top_h['low_pct']:.1f}% ของรีวิวต่ำ | {int(top_h['high_series'])} ซีรีส์")
print(f"รีวิวที่ไม่มี user_rating (ไม่ได้ใช้): {n_norating} จาก {len(df_all)} ({n_norating / len(df_all) * 100:.1f}%)")